# Image → 3D Scene (Stage B · Trellis)

Upload one photo → get a `scene.glb` with **real 3D generated assets**, each detected object reconstructed by Trellis and placed on a fitted ground plane.

### IMPORTANT — start clean
1. `Runtime → Change runtime type → T4 GPU`.
2. If you ran this before: `Runtime → Disconnect and delete runtime`, then reconnect. A polluted env breaks the install.

Colab's default PyTorch (2.11 / CUDA 12.8) is **too new** for Trellis's prebuilt deps (spconv, xformers, kaolin), so step 1 pins **torch 2.4.0 + cu121**.

In [ ]:
# 0. Confirm GPU
!nvidia-smi -L

## 1a. Pin a Trellis-compatible PyTorch (run first, alone)
If Colab pops up **"RESTART RUNTIME"** after this cell, click it, then continue at **1b** (do NOT re-run 1a).

In [ ]:
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())

## 1b. Install Trellis + deps (~15–20 min)
Builds the custom CUDA ops against the pinned torch. `flash-attn` is intentionally skipped (xformers is used instead).

In [ ]:
import os
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

![ -d /content/TRELLIS ] || git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
%cd /content/TRELLIS
# All Trellis ops EXCEPT flash-attn (which won't build here):
!. ./setup.sh --basic --xformers --spconv --mipgaussian --diffoctreerast --nvdiffrast --kaolin

# App-side deps for detection / depth / geometry.
!pip install -q "transformers>=4.44,<5" timm accelerate huggingface_hub trimesh xatlas scipy

# Sanity: these must all import for Trellis to run.
import torch, spconv, nvdiffrast.torch  # noqa: F401
print('OK: torch', torch.__version__, '| spconv + nvdiffrast import fine')

## 2. Get the pipeline code

In [ ]:
import sys
REPO = '/content/image-3d-pipeline'
BRANCH = 'claude/sweet-cori-kzkhyr'
![ -d {REPO} ] || git clone --branch {BRANCH} https://github.com/sanjanamani/image-3d-pipeline.git {REPO}
!cd {REPO} && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('pipeline code ready:', REPO)

## 3. Upload your photo

In [ ]:
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = '/content/' + next(iter(uploaded))
print('uploaded:', IMAGE_PATH)

## 4. Build the 3D scene
Detects objects → depth → Trellis per object → places each on the fitted ground. A few minutes per object on a T4.

In [ ]:
import os
os.chdir(REPO)
from scene_build import run_build
glb_path = run_build(IMAGE_PATH, output_dir='/content/outputs')
print('\nDONE ->', glb_path)

## 5. View inline + download

In [ ]:
import base64
from IPython.display import HTML, display

b64 = base64.b64encode(open(glb_path, 'rb').read()).decode()
display(HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{b64}"
  camera-controls auto-rotate shadow-intensity="1"
  style="width:100%;height:520px;background:#222;"></model-viewer>
'''))

from google.colab import files as _f
_f.download(glb_path)